# LAoN — Train YOLO artifact crop model (free Colab GPU)\n\nSocial / near-zero cost path:\n1. Label images as single class `artifact` (YOLO format)\n2. Zip `images/` + `labels/` and upload here\n3. Train YOLO11n-seg or detect\n4. Download `best.pt` → put in `ml/weights/` locally\n\n**Runtime → Change runtime type → GPU**

In [ ]:
!pip -q install ultralytics
import torch
print('cuda:', torch.cuda.is_available())

In [ ]:
# Upload a zip with structure:
# artifact/images/train, artifact/images/val, artifact/labels/train, artifact/labels/val
from google.colab import files
import zipfile, pathlib
uploaded = files.upload()
zname = next(iter(uploaded))
zipfile.ZipFile(zname).extractall('data')
print(list(pathlib.Path('data').rglob('*'))[:30])

In [ ]:
from pathlib import Path
yaml = Path('laon-artifact.yaml')
# Adjust 'path' if your extract layout differs
yaml.write_text('''
path: data/artifact
train: images/train
val: images/val
names:
  0: artifact
'''.strip())
print(yaml.read_text())

In [ ]:
from ultralytics import YOLO

# nano segment model — good CPU later; free Colab trains quickly
model = YOLO('yolo11n-seg.pt')
results = model.train(
    data='laon-artifact.yaml',
    epochs=80,
    imgsz=640,
    batch=16,
    project='laon',
    name='artifact-seg',
)
print(results)

In [ ]:
# Export ONNX for portable inference
best = 'laon/artifact-seg/weights/best.pt'
m = YOLO(best)
m.export(format='onnx', imgsz=640)
print('Download best.pt and best.onnx from laon/artifact-seg/weights/')
from google.colab import files
files.download(best)